# Fine-tuning do Assistente Virtual Médico (LoRA/QLoRA)

Rode este notebook em um runtime com GPU (**Runtime > Change runtime type > GPU**).
Ele reproduz `finetuning/train.py` e `finetuning/evaluate.py` do repositório.

**Antes de começar — token da Hugging Face (recomendado):**
sem ele os downloads são anônimos e compartilham o limite de taxa do IP do Colab,
ficando mais lentos e podendo falhar com HTTP 429 no meio dos 2 GB de pesos.

1. Crie um token de leitura em https://huggingface.co/settings/tokens
2. No Colab, abra **Secrets** (ícone 🔑 na barra lateral)
3. Adicione um secret chamado `HF_TOKEN` com esse valor
4. Ative **Notebook access** para ele

Os scripts detectam esse secret automaticamente. Para modelos de licença
restrita (Llama 3, Gemma), o token é obrigatório e é preciso aceitar os
termos na página do modelo antes.

In [ ]:
# Confirme que o runtime é GPU (Runtime > Change runtime type > T4 GPU).
# TPU não funciona: o QLoRA usa bitsandbytes, que só tem kernels CUDA/ROCm.
!nvidia-smi || echo 'SEM GPU — troque o runtime para T4 GPU antes de continuar.'

## Atualizando para a versão mais recente

Há **duas** coisas que podem estar desatualizadas, e elas são independentes:

1. **Este notebook.** O Colab carregou uma cópia do GitHub e não a
   atualiza sozinha. Para pegar a versão nova:
   *File > Open notebook > GitHub*, cole a URL do repositório e abra
   `finetuning/notebooks/colab_finetune.ipynb` de novo.
   Se você salvou uma cópia no Drive, ela é um arquivo separado e não
   recebe as atualizações.

2. **O código clonado na máquina do Colab** (`/content/...`). É o que
   realmente roda. A célula abaixo faz `git pull` quando o repositório já
   existe, então basta executá-la de novo.

Se algo ficar inconsistente, o caminho mais garantido é
*Runtime > Disconnect and delete runtime* e rodar tudo do zero: a máquina
é recriada limpa e o clone vem atualizado.


In [ ]:
# Clona na primeira execução e ATUALIZA nas seguintes.
# `git clone` falha se o diretório já existe, então esta célula é
# idempotente: pode ser rodada quantas vezes for preciso.
import os

REPO = "https://github.com/RenanAmaral/FIAP-9IADT-medical-agent-fine-tuned.git"
DIR = "/content/FIAP-9IADT-medical-agent-fine-tuned"

if os.path.isdir(os.path.join(DIR, ".git")):
    print("Repositório já existe — atualizando com git pull...")
    !cd {DIR} && git pull --ff-only
else:
    !git clone {REPO} {DIR}

%cd {DIR}
!pip install -q -r requirements.txt

# Confirma qual versão do código está carregada.
!git log --oneline -1


In [ ]:
# Confere se o token foi encontrado (não imprime o valor do token).
from finetuning.hf_auth import ensure_hf_login

ensure_hf_login()

In [ ]:
# O dataset já vem versionado em data/processed/.
# Rode esta célula apenas se quiser regenerá-lo do zero.
!python -m preprocessing.run_pipeline

## Sobreviver a desconexões (recomendado)

O Colab desconecta por inatividade ou por limite de uso, e quando a
máquina é reciclada tudo em `/content/` é perdido — inclusive os
checkpoints do treino.

Montar o Google Drive e gravar os checkpoints lá resolve isso: se a
sessão cair, os checkpoints continuam no Drive e o treino retoma de
onde parou com `--resume`.

Execute a célula abaixo **antes** do treino. Se preferir não usar o
Drive, pule — mas aí uma desconexão custa o treino inteiro.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Checkpoints e adapters passam a ser gravados no Drive.
OUTPUT_DIR = '/content/drive/MyDrive/tech-challenge-fase3/medical-assistant-lora'
print('Saída do treino:', OUTPUT_DIR)


## Treino

O script imprime o plano de treino antes de começar (exemplos, batch efetivo,
número de atualizações de peso) e avisa se forem poucas demais para produzir
um modelo mensuravelmente diferente do base.

Com os padrões atuais são ~168 atualizações, e o treino leva poucos minutos
numa T4.

In [ ]:
# --resume retoma do checkpoint mais recente se o treino foi interrompido;
# numa primeira execução ele simplesmente começa do zero.
!python -m finetuning.train \
    --base-model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --output-dir "{OUTPUT_DIR}" \
    --resume


In [ ]:
!python -m finetuning.evaluate \
    --base-model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --adapter-dir "{OUTPUT_DIR}" \
    --test-file data/processed/test.jsonl


In [ ]:
# Relatório de avaliação (perplexidade, ROUGE e comparação antes/depois)
from IPython.display import Markdown, display

display(Markdown(open('finetuning/eval_results/evaluation_report.md').read()))

In [ ]:
# Copia os adapters do Drive para o caminho padrão do repositório, que é
# onde assistant/llm.py os procura por padrão.
!mkdir -p finetuning/adapters
!cp -r "{OUTPUT_DIR}" finetuning/adapters/medical-assistant-lora

# Teste rápido do assistente completo já com o modelo fine-tuned
!python -m assistant.database
!python -m graphs.cli --backend finetuned --paciente PAC-0003 \
    --pergunta "Qual a conduta para este paciente?"


In [ ]:
# Empacota adapters e relatório de avaliação para levar ao repositório local.
# (Se você usou o Drive, eles já estão salvos lá — isto é só conveniência.)
!zip -qr resultados_finetuning.zip finetuning/adapters finetuning/eval_results
from google.colab import files

files.download('resultados_finetuning.zip')


## Próximo passo

Descompacte `resultados_finetuning.zip` na raiz do repositório local e commite
`finetuning/eval_results/`. Depois preencha a tabela de métricas do §3.6 de
`docs/relatorio_tecnico.md` com os números obtidos aqui.